In [1]:
# Libraries
import json
import numpy as np
import pandas as pd

pd.set_option('display.width', 200)
pd.set_option('display.max_columns', 20)

In [2]:
# Load the trained model and the period tables made by 01_train_linear_regression.ipynb
with open('trained_model.json') as f:
    trained = json.load(f)
models = trained['models']
tables = {level: pd.read_csv(f'data/{level}.csv') for level in ['yearly', 'monthly', 'weekly']}
TARGETS = {'product_performance': 'total_sales', 'customer_purchase_pattern': 'orders'}
print(trained['model'], '| trained on', trained['trained_on']['from'], 'to', trained['trained_on']['to'])

Simple Linear Regression (OLS), Y = a + bX | trained on 2021-11-01 to 2026-08-31


In [3]:
# Forecast: Y = a + bX
def predict(model, x):
    return model['constant'] + model['slope'] * x

# Next periods after the data: 2027, Sep 2026 (month no. 59), week of Sep 7, 2026 (week no. 254)
NEXT = {
    'yearly': [(2027, '2027')],
    'monthly': [(59 + i, (pd.Period('2026-09', 'M') + i).strftime('%b %Y')) for i in range(6)],
    'weekly': [(254 + i, f"Week of {pd.Timestamp('2026-09-07') + pd.Timedelta(weeks=i):%b %d, %Y}") for i in range(4)],
}
rows = []
for level, points in NEXT.items():
    for x, label in points:
        rows.append({
            'level': level, 'period': label, 'X': x,
            'product performance (PHP sales)': round(predict(models['product_performance'][level], x), 2),
            'customer purchase pattern (orders)': round(predict(models['customer_purchase_pattern'][level], x), 2),
        })
forecast = pd.DataFrame(rows)
forecast

,level,period,X,product performance (PHP sales),customer purchase pattern (orders)
0,yearly,2027,2027,452190.27,541.60
1,monthly,Sep 2026,59,35709.50,42.85
2,monthly,Oct 2026,60,35685.99,42.86
3,monthly,Nov 2026,61,35662.47,42.86
4,monthly,Dec 2026,62,35638.96,42.86
5,monthly,Jan 2027,63,35615.44,42.86
6,monthly,Feb 2027,64,35591.93,42.86
7,weekly,"Week of Sep 07, 2026",254,8085.14,9.71
8,weekly,"Week of Sep 14, 2026",255,8083.09,9.71
9,weekly,"Week of Sep 21, 2026",256,8081.04,9.70


In [4]:
# Accuracy check 1 - holdout: train on all but the last 6 periods, predict those 6, compare with the actual values
def fit_line(x, y):
    b, a = np.polyfit(x, y, 1)
    return a, b

def mape(actual, predicted):
    actual, predicted = np.asarray(actual, float), np.asarray(predicted, float)
    keep = actual > 0
    return float(np.mean(np.abs(actual[keep] - predicted[keep]) / actual[keep]) * 100)

HOLDOUT = 6
rows = []
for name, target in TARGETS.items():
    for level in ['monthly', 'weekly']:
        table = tables[level]
        if level == 'weekly':
            table = table.iloc[:-1]  # last week (Aug 31, 2026) has only one day of data
        train_part, test_part = table.iloc[:-HOLDOUT], table.iloc[-HOLDOUT:]
        a, b = fit_line(train_part['X'], train_part[target])
        error = mape(test_part[target], a + b * test_part['X'])
        rows.append({'model': name.replace('_', ' '), 'level': level, 'tested on': f'last {HOLDOUT} {level[:-2]}s',
                     'MAPE %': round(error, 2), 'accuracy %': round(100 - error, 2)})
holdout = pd.DataFrame(rows)
holdout

,model,level,tested on,MAPE %,accuracy %
0,product performance,monthly,last 6 months,10.49,89.51
1,product performance,weekly,last 6 weeks,12.15,87.85
2,customer purchase pattern,monthly,last 6 months,1.73,98.27
3,customer purchase pattern,weekly,last 6 weeks,2.84,97.16


In [5]:
# Accuracy check 2 - rolling: predict each month from all the months before it (at least 12 months of history)
rows = []
for name, target in TARGETS.items():
    table = tables['monthly']
    errors = []
    for i in range(12, len(table)):
        a, b = fit_line(table['X'][:i], table[target][:i])
        actual = table[target][i]
        if actual > 0:
            errors.append(abs(actual - (a + b * table['X'][i])) / actual * 100)
    rows.append({'model': name.replace('_', ' '), 'level': 'monthly', 'tests': len(errors),
                 'MAPE %': round(float(np.mean(errors)), 2), 'accuracy %': round(100 - float(np.mean(errors)), 2)})
rolling = pd.DataFrame(rows)
rolling

,model,level,tests,MAPE %,accuracy %
0,product performance,monthly,46,10.93,89.07
1,customer purchase pattern,monthly,46,2.41,97.59


In [6]:
# Accuracy check 3 - fit of the trained model on the complete periods (2021, 2026 and the last week are partial)
COMPLETE = {'yearly': slice(1, -1), 'monthly': slice(None), 'weekly': slice(None, -1)}
rows = []
for name, target in TARGETS.items():
    for level, table in tables.items():
        table = table.iloc[COMPLETE[level]]
        m = models[name][level]
        error = mape(table[target], predict(m, table['X']))
        rows.append({'model': name.replace('_', ' '), 'level': level, 'R2': round(m['r2'], 3), 'p-value': round(m['p'], 3),
                     'MAPE %': round(error, 2), 'accuracy %': round(100 - error, 2)})
fit_quality = pd.DataFrame(rows)
fit_quality

,model,level,R2,p-value,MAPE %,accuracy %
0,product performance,yearly,0.126,0.489,19.82,80.18
1,product performance,monthly,0.007,0.528,10.97,89.03
2,product performance,weekly,0.005,0.277,22.59,77.41
3,customer purchase pattern,yearly,0.152,0.445,19.50,80.50
4,customer purchase pattern,monthly,0.000,0.926,2.25,97.75
5,customer purchase pattern,weekly,0.010,0.121,2.88,97.12
